# Lab 5 - Introduction to SimPy
**TEB2103 Modelling and Simulation - Week 8**

---

## 1. Learning Outcomes

By the end of this lab, students should be able to:

- Understand **Discrete Event Simulation (DES)**
- Install and import **SimPy**
- Create a simulation **environment**
- Create and execute **processes**
- Simulate **events over time**
- Use multiple **concurrent processes**

**Documentation:** https://simpy.readthedocs.io/en/latest/index.html

## 2. Introduction

### What is SimPy?

**SimPy** is a Python library for **Discrete Event Simulation**, where the system only changes at specific points in time called **events**. The clock jumps straight from one event to the next instead of ticking through every moment in between.

SimPy is built on the generators from Lab 4. Each process is a generator function, and every `yield` hands an event to SimPy, which pauses the process until that event happens.

### Key terms

| Term | Meaning |
| :--- | :--- |
| `simpy.Environment()` | manages the simulation clock and schedules events |
| `env.now` | the current simulation time |
| `env.process()` | registers a generator function as a process |
| `env.timeout(t)` | an event that happens `t` time units from now |
| `simpy.Resource()` | a limited facility, e.g. a cashier, doctor or machine |
| `env.run(until=t)` | runs the simulation up to time `t` |

---
## 3. Guided Steps

### Step 1 - Install SimPy

> **Note:** Remove the `%` line if SimPy is already installed in your environment.

In [16]:
%pip install simpy

Note: you may need to restart the kernel to use updated packages.


### Step 2 - Import SimPy

In [17]:
import simpy
print("SimPy Version:", simpy.__version__)

SimPy Version: 4.1.2


### Step 3 - First SimPy Program

A SimPy simulation always starts by creating an **Environment**. The environment manages the simulation clock and schedules events.

In [18]:
import simpy

env = simpy.Environment()
print("Current Simulation Time:", env.now)

Current Simulation Time: 0


### Step 4 - Create Your First Process

A **process** is a Python function that models the behaviour of an entity. It must use `yield` to pause execution.

In [19]:
def clock(env):
    while True:
        print("Current Time:", env.now)
        yield env.timeout(1) #pause process for 1 time unit

env = simpy.Environment()
env.process(clock(env))
env.run(until=5)

Current Time: 0
Current Time: 1
Current Time: 2
Current Time: 3
Current Time: 4


### Step 5 - Two Processes Running Together

SimPy allows multiple processes to execute **concurrently**.

In [20]:
def clock(env, name, interval):
    while True:
        print(name, "Time =", env.now)
        yield env.timeout(interval)

env = simpy.Environment()
env.process(clock(env, "Fast Clock", 1))
env.process(clock(env, "Slow Clock", 2))
env.run(until=6)

Fast Clock Time = 0
Slow Clock Time = 0
Fast Clock Time = 1
Slow Clock Time = 2
Fast Clock Time = 2
Fast Clock Time = 3
Slow Clock Time = 4
Fast Clock Time = 4
Fast Clock Time = 5


### Step 6 - Customer Arrival Simulation

This example models a customer arriving and leaving after receiving service.

In [21]:
def customer(env, name):
    print(name, "arrives at", env.now)
    yield env.timeout(3)
    print(name, "leaves at", env.now)

env = simpy.Environment()
env.process(customer(env, "Ali"))
env.run()

Ali arrives at 0
Ali leaves at 3


### Step 7 - Multiple Customers

In [22]:
def customer(env, name, arrival):
    yield env.timeout(arrival)
    print(name, "arrived at", env.now)
    yield env.timeout(4)
    print(name, "left at", env.now)

env = simpy.Environment()
env.process(customer(env, "Ali", 1))
env.process(customer(env, "Ahmed", 3))
env.process(customer(env, "Sara", 5))
env.run()

Ali arrived at 1
Ahmed arrived at 3
Sara arrived at 5
Ali left at 5
Ahmed left at 7
Sara left at 9


### Step 8 - Resource Example (One Cashier)

A **Resource** represents a limited facility such as a cashier, doctor, or machine.

In [23]:
def customer(env, name, cashier):
    print(name, "arrives at", env.now)
    with cashier.request() as request:
        yield request
        print(name, "starts service at", env.now)
        yield env.timeout(3)
        print(name, "leaves at", env.now)

env = simpy.Environment()
cashier = simpy.Resource(env, capacity=1)
env.process(customer(env, "Ali", cashier))
env.process(customer(env, "Ahmed", cashier))
env.run()

Ali arrives at 0
Ahmed arrives at 0
Ali starts service at 0
Ali leaves at 3
Ahmed starts service at 3
Ahmed leaves at 6


### Step 9 - Resource with Arrival Times

In [24]:
def customer(env, name, cashier, arrival):
    yield env.timeout(arrival)
    print(name, "arrives at", env.now)
    with cashier.request() as request:
        yield request
        print(name, "starts service at", env.now)
        yield env.timeout(2)
        print(name, "leaves at", env.now)

env = simpy.Environment()
cashier = simpy.Resource(env, capacity=1)
env.process(customer(env, "Ali", cashier, 0))
env.process(customer(env, "Ahmed", cashier, 1))
env.process(customer(env, "Sara", cashier, 2))
env.run()

Ali arrives at 0
Ali starts service at 0
Ahmed arrives at 1
Sara arrives at 2
Ali leaves at 2
Ahmed starts service at 2
Ahmed leaves at 4
Sara starts service at 4
Sara leaves at 6


### Step 10 - Traffic Light Simulation

In [25]:
def traffic_light(env):
    while True:
        print("Green Light :", env.now)
        yield env.timeout(5)
        print("Yellow Light:", env.now)
        yield env.timeout(2)
        print("Red Light   :", env.now)
        yield env.timeout(5)

env = simpy.Environment()
env.process(traffic_light(env))
env.run(until=20)

Green Light : 0
Yellow Light: 5
Red Light   : 7
Green Light : 12
Yellow Light: 17
Red Light   : 19


---
## 4. Exercises

### Exercise 1
> Create a clock that prints every 2 time units.

**Explanation**

Firstly, the clock from Step 4 was changed to yield `env.timeout(2)` instead of `env.timeout(1)`, so the process pauses for 2 time units after every print. SimPy skips t = 1, 3 and 5 completely, as the clock jumps straight to the next event. With `env.run(until=10)`, the clock printed 5 times at t = 0, 2, 4, 6 and 8. The time t = 10 is not printed, because the simulation stops right at 10, similar to a shop closing exactly at 10 pm.

In [26]:
import simpy

def clock(env):
    while True:
        print("Current Time:", env.now)
        yield env.timeout(2)

env = simpy.Environment()
env.process(clock(env))
env.run(until=10)

Current Time: 0
Current Time: 2
Current Time: 4
Current Time: 6
Current Time: 8


### Exercise 2
> Create three clocks with intervals of 1, 2, and 3 seconds.

**Explanation**

Secondly, three processes were created from the same `clock()` function from Step 5, each with its own name and interval, and all three run together in one environment. Over 7 time units, the clocks ticked 7, 4 and 3 times. At t = 6, all three clocks tick at the same time, whereby SimPy runs them in the order they were scheduled. Hence, the Slow Clock prints first at t = 6, as it scheduled that tick earliest, at t = 3.

In [27]:
import simpy

def clock(env, name, interval):
    while True:
        print(name, "Time =", env.now)
        yield env.timeout(interval)

env = simpy.Environment()
env.process(clock(env, "Fast Clock", 1))
env.process(clock(env, "Medium Clock", 2))
env.process(clock(env, "Slow Clock", 3))
env.run(until=7)

Fast Clock Time = 0
Medium Clock Time = 0
Slow Clock Time = 0
Fast Clock Time = 1
Medium Clock Time = 2
Fast Clock Time = 2
Slow Clock Time = 3
Fast Clock Time = 3
Medium Clock Time = 4
Fast Clock Time = 4
Fast Clock Time = 5
Slow Clock Time = 6
Medium Clock Time = 6
Fast Clock Time = 6


### Exercise 3
> Simulate five customers arriving at a bank.

**Explanation**

Thirdly, the cashier example from Step 9 was extended to five customers, who arrive 1 time unit apart, while each service takes 3 time units. Since service is slower than arrivals, a queue builds up, much like a bank counter during lunch hour. `yield request` makes each customer wait until the cashier is free, and the waiting time is the start time minus the arrival time. As a result, Ali did not wait at all, while Daniel waited 8 time units, giving an average wait of 4.0.

In [28]:
import simpy

waits = []

def customer(env, name, cashier, arrival):
    yield env.timeout(arrival)
    print(name, "arrives at", env.now)
    with cashier.request() as request:
        yield request
        print(name, "starts service at", env.now)
        waits.append(env.now - arrival)
        yield env.timeout(3)
        print(name, "leaves at", env.now)

env = simpy.Environment()
cashier = simpy.Resource(env, capacity=1)
env.process(customer(env, "Ali", cashier, 0))
env.process(customer(env, "Ahmed", cashier, 1))
env.process(customer(env, "Sara", cashier, 2))
env.process(customer(env, "Aisyah", cashier, 3))
env.process(customer(env, "Daniel", cashier, 4))
env.run()

print("Waiting times:", waits)
print("Average wait:", sum(waits) / len(waits))

Ali arrives at 0
Ali starts service at 0
Ahmed arrives at 1
Sara arrives at 2
Aisyah arrives at 3
Ali leaves at 3
Ahmed starts service at 3
Daniel arrives at 4
Ahmed leaves at 6
Sara starts service at 6
Sara leaves at 9
Aisyah starts service at 9
Aisyah leaves at 12
Daniel starts service at 12
Daniel leaves at 15
Waiting times: [0, 2, 4, 6, 8]
Average wait: 4.0


### Exercise 4
> Modify the cashier example to have 2 cashiers instead of 1.

**Explanation**

Fourthly, the same five customers were simulated again, but with `capacity=2`, so two customers can be served at the same time. Ali and Ahmed went straight to a cashier, while the others only waited 1 or 2 time units. Hence, the average wait dropped from 4.0 to 0.8, and the last customer left at t = 9 instead of t = 15. The only change in the code is the capacity value, as SimPy handles the queue on its own.

In [29]:
import simpy

waits = []

def customer(env, name, cashier, arrival):
    yield env.timeout(arrival)
    print(name, "arrives at", env.now)
    with cashier.request() as request:
        yield request
        print(name, "starts service at", env.now)
        waits.append(env.now - arrival)
        yield env.timeout(3)
        print(name, "leaves at", env.now)

env = simpy.Environment()
cashier = simpy.Resource(env, capacity=2)     # 2 cashiers instead of 1
env.process(customer(env, "Ali", cashier, 0))
env.process(customer(env, "Ahmed", cashier, 1))
env.process(customer(env, "Sara", cashier, 2))
env.process(customer(env, "Aisyah", cashier, 3))
env.process(customer(env, "Daniel", cashier, 4))
env.run()

print("Waiting times:", waits)
print("Average wait:", sum(waits) / len(waits))

Ali arrives at 0
Ali starts service at 0
Ahmed arrives at 1
Ahmed starts service at 1
Sara arrives at 2
Aisyah arrives at 3
Ali leaves at 3
Sara starts service at 3
Daniel arrives at 4
Ahmed leaves at 4
Aisyah starts service at 4
Sara leaves at 6
Daniel starts service at 6
Aisyah leaves at 7
Daniel leaves at 9
Waiting times: [0, 0, 1, 1, 2]
Average wait: 0.8


### Exercise 5
> Create a traffic signal that runs for 30 simulation time units.

**Explanation**

Lastly, the traffic light from Step 10 was run with `env.run(until=30)`, where green lasts 5 time units, yellow 2 and red 5, so one cycle takes 12 time units. A counter was added to count each completed cycle. Within 30 time units, the light completed 2 full cycles, and the third cycle was cut off at yellow when the simulation stopped. This is similar to a traffic light at a junction, which keeps repeating until it is switched off.

In [30]:
import simpy

cycles = 0

def traffic_light(env):
    global cycles
    while True:
        print("Green Light :", env.now)
        yield env.timeout(5)
        print("Yellow Light:", env.now)
        yield env.timeout(2)
        print("Red Light   :", env.now)
        yield env.timeout(5)
        cycles += 1

env = simpy.Environment()
env.process(traffic_light(env))
env.run(until=30)

print("Full cycles completed:", cycles)

Green Light : 0
Yellow Light: 5
Red Light   : 7
Green Light : 12
Yellow Light: 17
Red Light   : 19
Green Light : 24
Yellow Light: 29
Full cycles completed: 2


---
## 5. Viva Questions

**1. What is SimPy?**

SimPy is a process-based Discrete Event Simulation library for Python. Each entity in the system, such as a customer or a machine, is written as a generator function called a process, and SimPy runs these processes along a shared simulation clock.

**2. What is Discrete Event Simulation?**

Discrete Event Simulation (DES) is a type of simulation where the state of the system only changes at specific points in time called events, such as a customer arriving or a service ending. Nothing happens between two events, so the clock jumps straight from one event to the next instead of moving in fixed steps.

**3. What is an Environment in SimPy?**

The Environment is the core of every SimPy simulation. It holds the simulation clock, keeps a list of scheduled events and decides which process runs next. It is created with `simpy.Environment()`.

**4. What does `env.now` return?**

`env.now` returns the current simulation time. It starts at 0 and only moves forward when the environment processes the next event. It is simulation time, not real time, so 10 time units can pass in a fraction of a second.

**5. Why is `yield` used in SimPy?**

`yield` is used to pause a process and hand an event to SimPy, such as a timeout or a resource request. SimPy resumes the process from the same line once that event has happened. This is the same pausing behaviour as the generators from Lab 4, which is why every SimPy process must be a generator function.

**6. What does `env.timeout()` do?**

`env.timeout(t)` creates an event that happens `t` time units after the current time. When a process yields it, the process is paused for that amount of time, which is used to model durations such as service time or how long a traffic light stays green.

**7. What is the purpose of `env.process()`?**

`env.process()` registers a generator as a process in the environment and schedules it to start. Calling the generator function alone, such as `clock(env)`, only creates the generator object, so nothing will run until it is passed to `env.process()`.

**8. What is a Resource?**

A Resource is a shared facility with a limited capacity, such as a cashier, doctor or machine. A process must request it before using it and release it after. It is created with `simpy.Resource(env, capacity=n)`, where `n` is how many processes can use it at the same time.

**9. What happens when multiple processes request the same resource?**

If the resource is already at full capacity, the extra requests are put in a queue and those processes wait. The queue is first come, first served, so when a user releases the resource, the next process in line is given it. This was shown in Exercise 3, where Daniel waited 8 time units behind four other customers.

**10. What is the difference between `env.run()` and `env.run(until=10)`?**

`env.run()` runs the simulation until there are no more events left, so it only ends on its own if every process finishes. A process with `while True`, like a clock, would make it run forever. `env.run(until=10)` stops the simulation at t = 10, even if there are still events scheduled after that, which is needed for processes that never end.

---
## 6. Summary

### Key Points
- SimPy uses **generators** as processes
- The **Environment** controls the simulation clock
- `env.timeout()` **pauses** a process for a duration
- **Resources** model limited facilities and queue requests automatically
- Multiple processes can run **concurrently** in one environment

### Connection to Modelling & Simulation

SimPy is a good way to model systems where entities **wait** for each other, such as:
**banks, hospitals, manufacturing lines and traffic systems**